In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

%md
## Ingestion del archivo country.json

In [0]:
country_df = (
    spark.read
        .format('json')
        .option('inferSchema', True)
        .load(f'{bronze_folder_path}/incremental_append/{v_file_date}/country.json')
)
display(country_df)
country_df.printSchema()

In [0]:
country_schema = "countryId string, countryIsoCode string, countryName string"
country_df = (
    spark.read
        .format('json')
        .option('header', True)
        .schema(country_schema)
        .load(f'{bronze_folder_path}/incremental_append/{v_file_date}/country.json')
)
display(country_df)
country_df.printSchema()

### Eliminar columnas

#### Metodo 1

In [0]:
country_df_dropped = country_df.drop('countryIsoCode')

display(country_df_dropped)

#### Metodo 2

In [0]:
country_df_dropped = country_df.drop(country_df['countryIsoCode'])

display(country_df_dropped)

#### Metodo 3

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp

In [0]:
country_df_dropped = country_df.drop(col('countryIsoCode'))

display(country_df_dropped)

In [0]:
country_df_final = (
    add_ingestion_date(country_df_dropped)
        .withColumnsRenamed({'countryId': 'country_id', 'countryName': 'country_name'})
        .withColumn('env', lit(v_env))
        .withColumn('file_date', lit(v_file_date))
)
display(country_df_final)

In [0]:
(
    country_df_final.write
        .mode('overwrite')
        .saveAsTable('movie_silver.countries')
)

In [0]:
dbutils.notebook.exit("Success")